In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
df = pd.read_csv('Telco-Customer-Churn-Cleaned.csv')

In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].mean(), inplace=True)

In [4]:
label_encoder = LabelEncoder()
df['Churn'] = label_encoder.fit_transform(df['Churn'])

In [5]:
df = pd.get_dummies(df, drop_first=True)

In [6]:
X = df.drop('Churn', axis=1)  # Replace 'Churn' with your target column name
y = df['Churn']

In [7]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [8]:
X_train = np.reshape(X_train, (X_train.shape[0], 1, X_train.shape[1]))
X_test = np.reshape(X_test, (X_test.shape[0], 1, X_test.shape[1]))

In [9]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [10]:
model = Sequential()
model.add(LSTM(units=64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.2))  # Prevent overfitting
model.add(LSTM(units=32))
model.add(Dropout(0.2))
model.add(Dense(units=2, activation='softmax'))

In [11]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [12]:
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=1)


Epoch 1/20
141/141 [==============================] - 9s 19ms/step - loss: 0.5092 - accuracy: 0.7659 - val_loss: 0.4280 - val_accuracy: 0.7986
Epoch 2/20
141/141 [==============================] - 1s 7ms/step - loss: 0.4345 - accuracy: 0.7930 - val_loss: 0.4183 - val_accuracy: 0.8030
Epoch 3/20
141/141 [==============================] - 1s 6ms/step - loss: 0.4274 - accuracy: 0.7959 - val_loss: 0.4113 - val_accuracy: 0.8039
Epoch 4/20
141/141 [==============================] - 1s 5ms/step - loss: 0.4254 - accuracy: 0.7974 - val_loss: 0.4079 - val_accuracy: 0.8030
Epoch 5/20
141/141 [==============================] - 1s 7ms/step - loss: 0.4203 - accuracy: 0.7992 - val_loss: 0.4062 - val_accuracy: 0.8092
Epoch 6/20
141/141 [==============================] - 1s 5ms/step - loss: 0.4184 - accuracy: 0.7990 - val_loss: 0.4058 - val_accuracy: 0.8075
Epoch 7/20
141/141 [==============================] - 1s 6ms/step - loss: 0.4146 - accuracy: 0.8065 - val_loss: 0.4086 - val_accuracy: 0.7950
Epoch

In [13]:
y_pred = np.argmax(model.predict(X_test), axis=1)
y_test_labels = np.argmax(y_test, axis=1)

print("Accuracy for LSTM:", accuracy_score(y_test_labels, y_pred))
print("\nClassification Report:\n", classification_report(y_test_labels, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_labels, y_pred))

45/45 [==============================] - 2s 2ms/step
Accuracy for LSTM: 0.8034066713981547

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.88      0.87      1036
           1       0.64      0.58      0.61       373

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409


Confusion Matrix:
 [[914 122]
 [155 218]]
